# 🌿 NDVI & MNDWI Report Generator with Google Earth Engine

**Automated Spectral Index Analysis for Mangrove & Peatland Community Groups (Kelompok)**

---

## Overview

This tutorial walks you through building an automated report generator that:

1. Connects to **Google Earth Engine (GEE)** and loads a polygon asset of community group boundaries
2. Pulls **Sentinel-2 satellite imagery** for one or more time periods
3. Computes **NDVI** (vegetation health) and **MNDWI** (surface water extent) indices
4. Classifies each index into meaningful land-cover categories
5. Calculates area statistics (hectares) per category
6. Downloads classified map thumbnails and generates bar charts
7. Compiles everything into a **polished multi-page PDF report**, one section per group per period

### Who is this for?

- Environmental analysts and GIS practitioners working with mangrove or peatland monitoring
- Python users who want to automate remote sensing workflows at scale
- Anyone who needs to produce standardized PDF reports from satellite data

### Prerequisites

- A **Google Earth Engine account** with an active Cloud project ([sign up here](https://earthengine.google.com/))
- A **GEE FeatureCollection asset** containing your study area polygons with a `kelompok` field (or any string field identifying each group)
- Basic familiarity with Python

---

## What are NDVI and MNDWI?

### NDVI — Normalized Difference Vegetation Index

NDVI measures the **density and health of green vegetation** using the contrast between near-infrared (NIR) light (strongly reflected by healthy plants) and red light (absorbed by chlorophyll):

$$NDVI = \frac{NIR - Red}{NIR + Red} = \frac{B8 - B4}{B8 + B4}$$

Values range from **−1 to +1**:

| Value Range | Interpretation |
|---|---|
| ≤ 0.0 | Water / Non-Vegetation (Kelas 1) |
| 0.0 – 0.2 | Bare Land / Very Sparse Vegetation (Kelas 2) |
| 0.2 – 0.5 | Moderate Vegetation (Kelas 3) |
| > 0.5 | Dense Vegetation / Healthy Mangrove (Kelas 4) |

### MNDWI — Modified Normalized Difference Water Index

MNDWI highlights **open water and wet areas** by exploiting the strong absorption of shortwave infrared (SWIR) by water:

$$MNDWI = \frac{Green - SWIR}{Green + SWIR}$$

We compute **two variants** using different SWIR bands:
- **MNDWI SWIR-1** uses Band 11 (1610 nm) — better for shallow water and mud flats
- **MNDWI SWIR-2** uses Band 12 (2190 nm) — better for deeper open water

| Value Range | Interpretation |
|---|---|
| ≤ −0.1 | Vegetation / Dryland (Kelas 1) |
| −0.1 – 0.0 | Transition Zone (Kelas 2) |
| 0.0 – 0.3 | Wet Area / Shallow Water / Tidal Mudflat (Kelas 3) |
| > 0.3 | Open Water (Kelas 4) |

---

## Workflow Diagram

```
GEE Asset (FeatureCollection)
        │
        ▼
For each Kelompok (group):
  └─ For each Time Period:
       ├─ [1] Filter Sentinel-2 → Cloud Mask → Median Composite
       ├─ [2] Compute NDVI, MNDWI-S1, MNDWI-S2
       ├─ [3] Classify each index into 4 classes
       ├─ [4] Compute area (ha) per class
       ├─ [5] Download map thumbnails (basemap + classified overlay)
       └─ [6] Generate bar charts
              │
              ▼
       PDF Report (one page per group per period)
```

---
## Step 1 — Install Dependencies

Install the required Python libraries. This only needs to be run once in a new environment (e.g., a fresh Google Colab session).

| Library | Purpose |
|---|---|
| `earthengine-api` | Connect to Google Earth Engine |
| `geemap` | Utility helpers for GEE (optional but useful) |
| `reportlab` | Build the PDF report |
| `matplotlib` | Generate map overlays and bar charts |
| `Pillow` | Image processing (compositing map layers) |
| `requests` | Download thumbnail images from GEE |
| `pandas` | Data manipulation (optional, listed for completeness) |

In [ ]:
# Install all required libraries
# In Colab you can run this cell once — the packages persist for the session
!pip install earthengine-api geemap reportlab matplotlib Pillow requests pandas

---
## Step 2 — Import Libraries

We import everything upfront so any missing package is caught immediately. Key imports:

- `ee` — the Earth Engine Python client
- `matplotlib` — used in **Agg** (non-interactive) mode so plots save to files rather than displaying inline
- `reportlab` — a powerful PDF layout engine (Platypus flowable model)
- `PIL` (Pillow) — composites the basemap and classified overlay images

In [ ]:
import ee
import os, sys, tempfile, textwrap, requests
from io import BytesIO
from datetime import datetime

import numpy as np
import pandas as pd

# Use the 'Agg' backend so matplotlib saves figures to files
# (no GUI window is opened — important for headless environments like Colab)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.ticker as ticker

from PIL import Image as PILImage

# ReportLab — PDF generation
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm, mm
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image as RLImage,
    Table, TableStyle, PageBreak, HRFlowable, KeepTogether
)

print("✓ All libraries imported successfully")

---
## Step 3 — Configuration

All user-editable settings live in a single `CONFIG` dictionary. **Edit this block to match your own data before running the rest of the notebook.**

### Key fields to update:

| Key | What to put here |
|---|---|
| `gee_project` | Your GEE Cloud Project ID (find it at [console.cloud.google.com](https://console.cloud.google.com)) |
| `prm_asset` | Full GEE path to your FeatureCollection (e.g. `projects/my-project/assets/MyPolygons`) |
| `kelompok_field` | The attribute field that names each group (used to split analysis by group) |
| `field_kabupaten / kecamatan / desa` | Administrative fields to print in the report header (set to `None` if they don't exist in your asset) |
| `periods` | List of time windows to analyse — add as many as you need |
| `output_pdf` | Name of the generated PDF file |
| `ndvi_thresholds` | NDVI class boundaries (the defaults match FAO mangrove guidelines) |
| `mndwi_thresholds` | MNDWI class boundaries |

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║                        CONFIG                           ║
# ╚══════════════════════════════════════════════════════════╝
#
# ⚠️  Edit the values below to match your project and data.

CONFIG = {
    # ── GEE Project ID ──────────────────────────────────────────────────────
    # Find at: https://console.cloud.google.com → select project → copy Project ID
    # Example: "ee-myusername"  or  "my-gee-project-123"
    "gee_project"      : "gee-muhammadrifqy",   # <-- CHANGE THIS

    # ── GEE FeatureCollection asset path ────────────────────────────────────
    # The polygon layer containing your study area boundaries, one feature per group
    "prm_asset"        : "projects/gee-muhammadrifqy/assets/PRM_2026_Riau_1936Ha",

    # ── Attribute that identifies each group ─────────────────────────────────
    # The analysis will loop over all unique values of this field
    "kelompok_field"   : "kelompok",

    # ── Administrative fields to display in the report header ────────────────
    # Set any of these to None if the field does not exist in your asset
    "field_kabupaten"  : "kabupaten",   # district / county
    "field_kecamatan"  : "kecamatan",   # sub-district
    "field_desa"       : "desa",        # village

    # ── Analysis periods ─────────────────────────────────────────────────────
    # Add or remove dicts to control which date ranges are processed.
    # The end date is EXCLUSIVE in GEE (i.e., images up to but not including that date).
    "periods": [
        {"label": "Tahun 2025", "start": "2025-01-01", "end": "2025-12-31"},
        {"label": "Tahun 2026", "start": "2026-01-01", "end": "2026-03-10"},
    ],

    # ── Output PDF path ──────────────────────────────────────────────────────
    "output_pdf"       : "report_ndvi_mndwi.pdf",

    # ── Map thumbnail size (pixels) ──────────────────────────────────────────
    # GEE free tier allows up to 1280×1280 pixels.
    # Larger thumbnails = higher quality but slower download.
    "thumb_width"      : 512,
    "thumb_height"     : 512,

    # ── NDVI classification thresholds ───────────────────────────────────────
    # Defines 4 classes using boundary values:
    #   Class 1: index ≤ 0.0   → Water / Non-Vegetation
    #   Class 2: 0.0–0.2       → Bare Land
    #   Class 3: 0.2–0.5       → Moderate Vegetation
    #   Class 4: > 0.5         → Dense Vegetation
    "ndvi_thresholds"  : [-9999, 0.0, 0.2, 0.5, 9999],

    # ── MNDWI classification thresholds ──────────────────────────────────────
    #   Class 1: index ≤ −0.1  → Vegetation / Dryland
    #   Class 2: −0.1–0.0      → Transition Zone
    #   Class 3: 0.0–0.3       → Wet Area / Shallow Water
    #   Class 4: > 0.3         → Open Water
    "mndwi_thresholds" : [-9999, -0.1, 0.0, 0.3, 9999],
}

print("✓ Configuration loaded")
print(f"  Project  : {CONFIG['gee_project']}")
print(f"  Asset    : {CONFIG['prm_asset']}")
print(f"  Periods  : {[p['label'] for p in CONFIG['periods']]}")
print(f"  Output   : {CONFIG['output_pdf']}")

---
## Step 4 — Define Legend Classes

The legend dictionaries define the visual appearance (color) and label for each class. These are used in:
- The map overlay (GEE visualization palette)
- Matplotlib legends and bar charts
- The PDF cover page legend

> **Tip:** If you change the thresholds in `CONFIG`, also update the labels here to match.

In [ ]:
# ── NDVI Legend ──────────────────────────────────────────────────────────────
# Colors progress from red (non-vegetation) to dark green (dense vegetation)
NDVI_LEGEND = [
    {"id": 1, "label": "Air / Non Vegetasi",  "hex": "#FF0000", "rgb": (1.0, 0.0, 0.0)},
    {"id": 2, "label": "Lahan Terbuka",        "hex": "#BFFF00", "rgb": (0.75, 1.0, 0.0)},
    {"id": 3, "label": "Vegetasi Sedang",      "hex": "#4CBB17", "rgb": (0.30, 0.73, 0.09)},
    {"id": 4, "label": "Vegetasi Rapat",       "hex": "#006400", "rgb": (0.0, 0.39, 0.0)},
]

# ── MNDWI Legend ─────────────────────────────────────────────────────────────
# Colors progress from light blue (dryland) to dark navy (open water)
MNDWI_LEGEND = [
    {"id": 1, "label": "Vegetasi / Daratan",
                                              "hex": "#B0E0FF", "rgb": (0.69, 0.88, 1.0)},
    {"id": 2, "label": "Zona Transisi",       "hex": "#5B9BD5", "rgb": (0.36, 0.61, 0.84)},
    {"id": 3, "label": "Area Basah / Perairan Dangkal / Lumpur Pasang Surut",
                                              "hex": "#2E4FA3", "rgb": (0.18, 0.31, 0.64)},
    {"id": 4, "label": "Perairan Terbuka",    "hex": "#1A0A5E", "rgb": (0.10, 0.04, 0.37)},
]

print("NDVI Classes:")
for c in NDVI_LEGEND:
    print(f"  {c['id']}. {c['label']:30s}  {c['hex']}")

print("\nMNDWI Classes:")
for c in MNDWI_LEGEND:
    print(f"  {c['id']}. {c['label']:55s}  {c['hex']}")

---
## Step 5 — Google Earth Engine Functions

This section defines all the GEE processing functions. Read through the docstrings — each function has a single, focused job.

### 5.1 Authenticate & Initialize GEE

GEE authentication uses OAuth2. On first run it opens a browser to generate a token. Subsequent runs use the cached token automatically.

In [ ]:
def init_gee():
    """
    Initialize the Earth Engine API.

    Tries ee.Initialize() first (uses cached credentials).
    If no credentials are found, runs ee.Authenticate() to open the browser
    OAuth flow, then initializes again.
    """
    project = CONFIG["gee_project"]
    try:
        ee.Initialize(project=project)
        print(f"✓ GEE initialized  (project: {project})")
    except Exception:
        print("! Credentials not found — launching ee.Authenticate() ...")
        print("  Follow the browser link, copy the token, and paste it below.")
        ee.Authenticate()
        ee.Initialize(project=project)
        print(f"✓ GEE initialized after authentication  (project: {project})")

### 5.2 Cloud Masking

Sentinel-2 SR products include a **Scene Classification Layer (SCL)** band that labels each pixel. We mask out clouds, cloud shadows, cirrus, and snow using these SCL codes:

| SCL Code | Meaning | Action |
|---|---|---|
| 3 | Cloud shadow | **Masked out** |
| 7 | Unclassified | **Masked out** |
| 8 | Cloud medium probability | **Masked out** |
| 9 | Cloud high probability | **Masked out** |
| 10 | Thin cirrus | **Masked out** |

We also divide reflectance values by 10,000 to convert from the integer DN format stored by GEE to **surface reflectance (0–1)**.

In [ ]:
def cloud_mask(image):
    """
    Apply cloud and cloud-shadow masking using the SCL band.

    SCL classes removed: 3 (cloud shadow), 7-10 (various cloud types).
    Reflectance bands are scaled from DN (0–10000) to reflectance (0–1).

    Returns: masked and scaled image (B-bands only)
    """
    scl  = image.select('SCL')
    # Build a boolean mask: True = clear pixel, False = cloud/shadow
    mask = scl.eq(3).Or(scl.gte(7).And(scl.lte(10))).eq(0)
    return image.select(['B.*']).divide(10000).updateMask(mask)


def build_sentinel2(geometry, start_date, end_date):
    """
    Build a cloud-free Sentinel-2 median composite for a given area and period.

    Steps:
      1. Filter the Sentinel-2 SR Harmonized collection by date and bounds
      2. Apply cloud masking to each image
      3. Compute the pixel-wise median across all valid observations
      4. Clip the result to the study area geometry

    The median reducer is robust to remaining cloud artefacts:
    if most observations are clear, the median will be a clear pixel.

    Args:
        geometry   : ee.Geometry — the study area
        start_date : str — 'YYYY-MM-DD'
        end_date   : str — 'YYYY-MM-DD' (exclusive)

    Returns: ee.Image (Sentinel-2 median composite)
    """
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterDate(start_date, end_date)
          .filterBounds(geometry)
          .map(cloud_mask)          # apply cloud mask to every image
          .median()                 # pixel-wise median across time
          .clip(geometry))          # restrict to study area
    return s2

### 5.3 Index Computation

We compute three spectral indices from the Sentinel-2 band arithmetic. All GEE operations here are **lazy** — they define a computation graph but don't run until `.getInfo()` or `.getThumbURL()` is called.

In [ ]:
def compute_indices(s2):
    """
    Compute NDVI and both MNDWI variants from a Sentinel-2 composite.

    Sentinel-2 bands used:
        B3  = Green (560 nm)      — for MNDWI
        B4  = Red   (665 nm)      — for NDVI
        B8  = NIR   (842 nm)      — for NDVI
        B11 = SWIR-1 (1610 nm)   — for MNDWI variant 1
        B12 = SWIR-2 (2190 nm)   — for MNDWI variant 2

    Returns:
        ndvi     : ee.Image — NDVI band
        mndwi_s1 : ee.Image — MNDWI using SWIR-1
        mndwi_s2 : ee.Image — MNDWI using SWIR-2
    """
    nir   = s2.select('B8')
    red   = s2.select('B4')
    green = s2.select('B3')
    swir1 = s2.select('B11')
    swir2 = s2.select('B12')

    # NDVI = (NIR − Red) / (NIR + Red)
    ndvi     = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

    # MNDWI (SWIR-1) = (Green − SWIR1) / (Green + SWIR1)
    mndwi_s1 = green.subtract(swir1).divide(green.add(swir1)).rename('MNDWI_S1')

    # MNDWI (SWIR-2) = (Green − SWIR2) / (Green + SWIR2)
    mndwi_s2 = green.subtract(swir2).divide(green.add(swir2)).rename('MNDWI_S2')

    return ndvi, mndwi_s1, mndwi_s2

### 5.4 Classification

The `classify_index` function converts a continuous floating-point index image into a discrete 4-class integer image. It does this using conditional GEE operations rather than a lookup table, which keeps everything server-side and avoids downloading pixel data.

In [ ]:
def classify_index(index_image, thresholds):
    """
    Classify a continuous spectral index into 4 integer classes (1–4).

    How it works:
        For each class boundary pair (t[i], t[i+1]), a boolean mask is
        created and multiplied by the class ID (1, 2, 3, or 4).
        The four masks are additive and non-overlapping, so the result
        is a single-band image with values 1–4.

    Args:
        index_image : ee.Image    — the NDVI or MNDWI image
        thresholds  : list[float] — 5 values defining 4 class boundaries
                                    e.g. [-9999, 0.0, 0.2, 0.5, 9999]

    Returns: ee.Image with band 'class' (integer values 1–4)
    """
    t = thresholds  # shorthand: [min, t1, t2, t3, max]
    classified = (
        index_image.gt(t[0]).And(index_image.lte(t[1])).multiply(1)   # Class 1
        .add(index_image.gt(t[1]).And(index_image.lte(t[2])).multiply(2))  # Class 2
        .add(index_image.gt(t[2]).And(index_image.lte(t[3])).multiply(3))  # Class 3
        .add(index_image.gt(t[3]).multiply(4))                         # Class 4
    )
    # Preserve the mask from the input image (masked pixels stay masked)
    return classified.rename('class').updateMask(index_image.mask())

### 5.5 Area Calculation

GEE's `ee.Image.pixelArea()` returns an image where each pixel's value is its area in square metres. By masking this to a single class and summing, we get the total area of that class within the study boundary.

> **Note:** We use `scale=10` (Sentinel-2 native resolution). Using a coarser scale would be faster but less accurate for small polygons.

In [ ]:
def compute_class_areas(classified_image, geometry, legend):
    """
    Calculate the area (in hectares) for each class within a geometry.

    Method:
        1. Create a pixel-area image (m² per pixel)
        2. For each class ID, mask the area image to that class only
        3. Sum the remaining pixel areas using reduceRegion
        4. Convert m² → ha (divide by 10,000)

    Args:
        classified_image : ee.Image   — output of classify_index()
        geometry         : ee.Geometry
        legend           : list[dict] — NDVI_LEGEND or MNDWI_LEGEND

    Returns: list of dicts [{id, label, hex, rgb, area_ha}, ...]
    """
    # pixelArea() in m², divide by 10,000 to get hectares
    area_image = ee.Image.pixelArea().divide(10000)

    results = []
    for cls in legend:
        cid = cls["id"]
        # Mask the area image so only pixels belonging to this class contribute
        mask     = classified_image.eq(cid)
        area_dict = (area_image.updateMask(mask)
                     .reduceRegion(
                         reducer   = ee.Reducer.sum(),
                         geometry  = geometry,
                         scale     = 10,        # Sentinel-2 native resolution
                         maxPixels = 1e13       # allow large regions
                     ).getInfo())
        area_ha = list(area_dict.values())[0] or 0.0
        results.append({
            "id"     : cid,
            "label"  : cls["label"],
            "hex"    : cls["hex"],
            "rgb"    : cls["rgb"],
            "area_ha": round(area_ha, 2),
        })
    return results

### 5.6 Thumbnail Download

GEE's `getThumbURL()` renders a map view as a PNG and returns a signed URL. We then download this image with the `requests` library and open it as a PIL Image for compositing.

In [ ]:
def get_thumb_url(image, geometry, vis_params, thumb_size):
    """
    Build and return a GEE thumbnail URL for a given image and geometry.

    Args:
        image      : ee.Image       — the image to render
        geometry   : ee.Geometry    — defines the bounding box
        vis_params : dict           — GEE visualization params (bands, min, max, palette)
        thumb_size : int            — width = height in pixels (e.g. 512)

    Returns: str — signed thumbnail URL (valid for ~1 hour)
    """
    region = geometry.bounds().getInfo()["coordinates"]
    params = dict(vis_params)          # copy so we don't mutate the caller's dict
    params["region"]     = region
    params["dimensions"] = f"{thumb_size}x{thumb_size}"
    params["format"]     = "png"
    return image.getThumbURL(params)


def download_image(url):
    """
    Download a PNG from a URL and return it as a PIL RGBA Image.

    RGBA mode is used so we can composite layers with transparency.
    """
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()   # raise an error on HTTP 4xx/5xx
    return PILImage.open(BytesIO(resp.content)).convert("RGBA")


def get_bounds(geometry):
    """
    Extract (lon_min, lon_max, lat_min, lat_max) from an ee.Geometry.

    Used to set the geographic extent of matplotlib axes so tick marks
    show real-world coordinates rather than pixel offsets.
    """
    coords = geometry.bounds().getInfo()["coordinates"][0]
    lons   = [c[0] for c in coords]
    lats   = [c[1] for c in coords]
    return min(lons), max(lons), min(lats), max(lats)

---
## Step 6 — Matplotlib Visualization Helpers

These functions produce the PNG files that are later embedded in the PDF.

### 6.1 Map with Classified Overlay

We composite two layers:
1. **Basemap** — a true-color (RGB = B4/B3/B2) Sentinel-2 image
2. **Classified overlay** — the NDVI or MNDWI class image, rendered semi-transparent so the satellite context shows through

Coordinate ticks are set from the actual geographic bounds so the axes display longitude/latitude values.

In [ ]:
def save_map_with_legend(basemap_img, classified_img, legend, title, bounds, out_path,
                         overlay_alpha=0.65):
    """
    Save a map PNG compositing a true-color basemap and a classified overlay.

    Layout:
        - Basemap (Sentinel-2 true colour) as background
        - Classified index overlay at 65% opacity so basemap shows through
        - Geographic coordinate grid (longitude/latitude tick marks)
        - Legend in the lower-left corner
        - North arrow in the upper-right corner

    Args:
        basemap_img    : PIL Image — true-colour Sentinel-2 thumbnail
        classified_img : PIL Image — classified index thumbnail (coloured by class)
        legend         : list[dict] — NDVI_LEGEND or MNDWI_LEGEND
        title          : str
        bounds         : tuple (lon_min, lon_max, lat_min, lat_max)
        out_path       : str — output file path (.png)
        overlay_alpha  : float — opacity of the classified layer (0–1)
    """
    lon_min, lon_max, lat_min, lat_max = bounds

    fig, ax = plt.subplots(figsize=(6, 6), dpi=140)

    # ── Layer 1: Basemap (true-colour satellite image) ──────────────────────
    ax.imshow(np.array(basemap_img),
              extent=[lon_min, lon_max, lat_min, lat_max],
              aspect='auto', origin='upper', interpolation='bilinear')

    # ── Layer 2: Classified overlay (semi-transparent) ─────────────────────
    overlay = np.array(classified_img.convert("RGBA")).copy().astype(float)
    overlay[..., 3] = overlay[..., 3] * overlay_alpha   # scale alpha channel
    ax.imshow(overlay.astype(np.uint8),
              extent=[lon_min, lon_max, lat_min, lat_max],
              aspect='auto', origin='upper', interpolation='nearest')

    # ── Coordinate grid ────────────────────────────────────────────────────
    import matplotlib.ticker as mticker
    n_ticks   = 4
    lon_ticks = np.linspace(lon_min, lon_max, n_ticks)
    lat_ticks = np.linspace(lat_min, lat_max, n_ticks)

    ax.set_xticks(lon_ticks)
    ax.set_yticks(lat_ticks)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.4f°'))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.4f°'))
    ax.tick_params(axis='both', labelsize=6.5, color='#555555',
                   labelcolor='#222222', length=3, width=0.6)
    ax.grid(True, linestyle='--', linewidth=0.45, color='white', alpha=0.6)

    ax.set_xlabel('Bujur (°)', fontsize=7.5, labelpad=3)
    ax.set_ylabel('Lintang (°)', fontsize=7.5, labelpad=3)

    for spine in ax.spines.values():
        spine.set_linewidth(0.6)
        spine.set_edgecolor('#444444')

    # ── Title ──────────────────────────────────────────────────────────────
    ax.set_title(title, fontsize=10, fontweight='bold', pad=7, color='#1a3c5e')

    # ── Legend (lower left) ────────────────────────────────────────────────
    patches    = [mpatches.Patch(color=c["hex"], label=c["label"]) for c in legend]
    legend_obj = ax.legend(
        handles=patches,
        loc='lower left', fontsize=6.5,
        framealpha=0.88, frameon=True,
        edgecolor='#888888', fancybox=False,
        title='Legenda', title_fontsize=7,
    )
    legend_obj.get_frame().set_linewidth(0.6)

    # ── North arrow (upper right) ──────────────────────────────────────────
    ax.annotate('', xy=(0.96, 0.96), xytext=(0.96, 0.88),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='white', lw=1.8))
    ax.text(0.96, 0.98, 'N', transform=ax.transAxes,
            ha='center', va='bottom', fontsize=8, fontweight='bold', color='white')

    plt.tight_layout(pad=0.8)
    fig.savefig(out_path, dpi=140, bbox_inches='tight')
    plt.close(fig)   # release memory


def save_area_barchart(area_data, title, out_path):
    """
    Save a horizontal bar chart showing the area (ha) of each class.

    Bar colors match the legend so the chart is immediately readable
    alongside the map. Value labels are printed to the right of each bar.

    Args:
        area_data : list[dict] — output of compute_class_areas()
        title     : str
        out_path  : str — output file path (.png)
    """
    labels = [textwrap.fill(d["label"], 28) for d in area_data]  # wrap long labels
    values = [d["area_ha"] for d in area_data]
    clrs   = [d["hex"] for d in area_data]

    fig, ax = plt.subplots(figsize=(5.5, 2.8), dpi=120)
    bars    = ax.barh(labels, values, color=clrs, edgecolor='white', height=0.55)

    ax.set_xlabel('Luas (ha)', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.tick_params(axis='y', labelsize=8)
    ax.tick_params(axis='x', labelsize=8)
    ax.invert_yaxis()   # Class 1 at the top
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)

    # Print exact value to the right of each bar
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + max(values) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{val:,.2f} ha', va='center', fontsize=7.5)

    plt.tight_layout(pad=0.8)
    fig.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close(fig)

---
## Step 7 — PDF Builder

ReportLab uses a **flowable** model: you build a list (`story`) of elements — paragraphs, tables, images, spacers — and then `doc.build(story)` lays them onto pages automatically.

### 7.1 Paragraph Styles

Custom styles inherit from ReportLab's built-in stylesheet and override properties like font size, color, and alignment.

In [ ]:
# ── Page dimensions ─────────────────────────────────────────────────────────
PAGE_W, PAGE_H = A4          # 595 × 842 points
MARGIN         = 1.8 * cm    # uniform page margin


def build_styles():
    """
    Create and return a dict of custom ReportLab ParagraphStyles.

    Styles defined:
        title        — report title (large, centered, navy)
        subtitle     — sub-title (medium, centered, steel blue)
        section      — section heading (bold, navy)
        body         — normal paragraph text
        table_header — white text for table header cells
        table_cell   — left-aligned table body cells
        footer       — small grey footer text
    """
    styles = getSampleStyleSheet()
    custom = {
        "title": ParagraphStyle(
            "CTitle", parent=styles["Title"],
            fontSize=18, spaceAfter=6,
            textColor=colors.HexColor("#1a3c5e"),
            alignment=TA_CENTER,
        ),
        "subtitle": ParagraphStyle(
            "CSubtitle", parent=styles["Normal"],
            fontSize=10, spaceAfter=4,
            textColor=colors.HexColor("#3d6b8e"),
            alignment=TA_CENTER,
        ),
        "section": ParagraphStyle(
            "CSection", parent=styles["Heading2"],
            fontSize=12, spaceAfter=4, spaceBefore=10,
            textColor=colors.HexColor("#1a3c5e"),
            borderPad=(0, 0, 2, 0),
        ),
        "body": ParagraphStyle(
            "CBody", parent=styles["Normal"],
            fontSize=9, spaceAfter=3, leading=14,
        ),
        "table_header": ParagraphStyle(
            "CTH", parent=styles["Normal"],
            fontSize=9, textColor=colors.white, alignment=TA_CENTER,
        ),
        "table_cell": ParagraphStyle(
            "CTC", parent=styles["Normal"],
            fontSize=9, alignment=TA_LEFT,
        ),
        "footer": ParagraphStyle(
            "CFooter", parent=styles["Normal"],
            fontSize=8, textColor=colors.grey, alignment=TA_CENTER,
        ),
    }
    return custom

print("✓ PDF style builder defined")

### 7.2 Area Statistics Table

Each index block includes an area table with columns: Class ID, Class Label, Area (ha), and Proportion (%). The total row at the bottom auto-sums all classes.

In [ ]:
def area_table(area_data, styles):
    """
    Build a styled ReportLab Table showing area statistics.

    Columns:
        #            — class ID
        Kelas        — class label
        Luas (ha)    — area in hectares
        Proporsi (%) — percentage of total area

    The last row shows the totals.
    Alternating row backgrounds aid readability.

    Args:
        area_data : list[dict] — output of compute_class_areas()
        styles    : dict       — output of build_styles()

    Returns: reportlab.platypus.Table
    """
    total  = sum(d["area_ha"] for d in area_data)

    # Header row
    header = [
        Paragraph("<b>#</b>",            styles["table_header"]),
        Paragraph("<b>Kelas</b>",        styles["table_header"]),
        Paragraph("<b>Luas (ha)</b>",    styles["table_header"]),
        Paragraph("<b>Proporsi (%)</b>", styles["table_header"]),
    ]
    rows = [header]

    # Data rows
    for d in area_data:
        pct = (d["area_ha"] / total * 100) if total > 0 else 0
        rows.append([
            Paragraph(str(d["id"]),        styles["table_cell"]),
            Paragraph(d["label"],          styles["table_cell"]),
            Paragraph(f'{d["area_ha"]:,.2f}', styles["table_cell"]),
            Paragraph(f'{pct:.1f}%',       styles["table_cell"]),
        ])

    # Total row
    rows.append([
        Paragraph("",                          styles["table_cell"]),
        Paragraph("<b>Total</b>",             styles["table_cell"]),
        Paragraph(f'<b>{total:,.2f}</b>',     styles["table_cell"]),
        Paragraph("<b>100%</b>",              styles["table_cell"]),
    ])

    col_widths = [1.0*cm, 8.5*cm, 3.0*cm, 3.0*cm]
    t = Table(rows, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        # Header styling
        ('BACKGROUND',     (0, 0),  (-1, 0),  colors.HexColor("#1a3c5e")),
        ('TEXTCOLOR',      (0, 0),  (-1, 0),  colors.white),
        ('ALIGN',          (0, 0),  (-1, -1), 'CENTER'),
        ('ALIGN',          (1, 1),  (1, -1),  'LEFT'),   # class labels are left-aligned
        ('FONTSIZE',       (0, 0),  (-1, -1), 9),
        # Alternating row backgrounds
        ('ROWBACKGROUNDS', (0, 1),  (-1, -2),
         [colors.HexColor("#f0f4f8"), colors.white]),
        # Total row
        ('BACKGROUND',     (0, -1), (-1, -1), colors.HexColor("#dce6f0")),
        ('LINEBELOW',      (0, 0),  (-1, 0),  0.5, colors.white),
        ('LINEABOVE',      (0, -1), (-1, -1), 0.8, colors.HexColor("#1a3c5e")),
        ('BOX',            (0, 0),  (-1, -1), 0.5, colors.HexColor("#aec6cf")),
        ('INNERGRID',      (0, 0),  (-1, -1), 0.25, colors.HexColor("#d0dde8")),
        ('TOPPADDING',     (0, 0),  (-1, -1), 4),
        ('BOTTOMPADDING',  (0, 0),  (-1, -1), 4),
        ('LEFTPADDING',    (0, 0),  (-1, -1), 5),
    ]))
    return t

### 7.3 Cover Page

The cover page gives a high-level summary of the report: data source, periods, resolution, number of groups, and legend definitions for both NDVI and MNDWI.

In [ ]:
def build_cover_page(story, styles, kelompok_list):
    """
    Append the cover page flowables to the story list.

    Includes:
        - Report title and subtitle
        - Metadata table (data source, periods, resolution, group count, date)
        - NDVI and MNDWI legend overviews
        - PageBreak so content starts on the next page
    """
    story.append(Spacer(1, 3*cm))
    story.append(Paragraph("LAPORAN ANALISIS INDEKS SPEKTRAL", styles["title"]))
    story.append(Paragraph("NDVI &amp; MNDWI — Lahan Gambut", styles["subtitle"]))
    story.append(Spacer(1, 0.3*cm))
    story.append(HRFlowable(width="80%", thickness=2,
                             color=colors.HexColor("#1a3c5e"), spaceAfter=10))
    story.append(Spacer(1, 0.5*cm))

    # Metadata table
    meta = [
        ["Sumber Data",     "Sentinel-2 SR Harmonized (COPERNICUS/S2_SR_HARMONIZED)"],
        ["Periode",         "  |  ".join(
            f'{p["label"]}: {p["start"]} – {p["end"]}' for p in CONFIG["periods"]
        )],
        ["Resolusi",        "10 m"],
        ["Jumlah Kelompok", str(len(kelompok_list))],
        ["Dibuat",          datetime.today().strftime("%d %B %Y")],
    ]
    t = Table(meta, colWidths=[4.5*cm, 12*cm])
    t.setStyle(TableStyle([
        ('FONTSIZE',       (0, 0), (-1, -1), 9),
        ('FONTNAME',       (0, 0), (0, -1), 'Helvetica-Bold'),
        ('VALIGN',         (0, 0), (-1, -1), 'TOP'),
        ('TOPPADDING',     (0, 0), (-1, -1), 4),
        ('BOTTOMPADDING',  (0, 0), (-1, -1), 4),
        ('LINEBELOW',      (0, 0), (-1, -2), 0.3, colors.HexColor("#d0dde8")),
    ]))
    story.append(t)
    story.append(Spacer(1, 1*cm))

    # NDVI legend overview
    story.append(Paragraph("Klasifikasi NDVI", styles["section"]))
    for c in NDVI_LEGEND:
        story.append(Paragraph(
            f'<font color="{c["hex"]}">&#9632;</font>  <b>Kelas {c["id"]}:</b>  {c["label"]}',
            styles["body"]
        ))

    story.append(Spacer(1, 0.4*cm))

    # MNDWI legend overview
    story.append(Paragraph("Klasifikasi MNDWI", styles["section"]))
    for c in MNDWI_LEGEND:
        story.append(Paragraph(
            f'<font color="{c["hex"]}">&#9632;</font>  <b>Kelas {c["id"]}:</b>  {c["label"]}',
            styles["body"]
        ))

    story.append(PageBreak())   # force a new page before the data pages

print("✓ PDF builder functions defined")

### 7.4 Per-Kelompok Pages

Each group's section contains:
- A header with the group name and administrative location
- One sub-section per analysis period
- Within each period: three index blocks (NDVI, MNDWI SWIR-1, MNDWI SWIR-2)
- Each block: a side-by-side map and bar chart, followed by the area statistics table

In [ ]:
def _add_index_block(story, styles, number, index_name, map_path, chart_path, area_data):
    """
    Append one index section (map + chart side-by-side + area table) to the story.

    Args:
        number     : int — section number (1, 2, or 3)
        index_name : str — display name (e.g. 'NDVI', 'MNDWI SWIR-1')
        map_path   : str — path to the map PNG
        chart_path : str — path to the bar chart PNG
        area_data  : list[dict]
    """
    map_w   = 8.0 * cm
    chart_w = 8.5 * cm

    story.append(Paragraph(f"{number}. {index_name}", styles["section"]))

    # Map and chart displayed side-by-side in a 1-row, 2-column table
    row = Table(
        [[RLImage(map_path,   width=map_w,   height=map_w),
          RLImage(chart_path, width=chart_w, height=4.5*cm)]],
        colWidths=[map_w + 0.3*cm, chart_w + 0.3*cm]
    )
    row.setStyle(TableStyle([
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('ALIGN',  (0, 0), (-1, -1), 'CENTER'),
    ]))
    story.append(row)
    story.append(Spacer(1, 0.3*cm))
    story.append(area_table(area_data, styles))
    story.append(Spacer(1, 0.5*cm))


def build_kelompok_page(story, styles, name, period_label,
                        ndvi_area, mndwi_s1_area, mndwi_s2_area,
                        ndvi_map_path, mndwi_s1_map_path, mndwi_s2_map_path,
                        ndvi_chart_path, mndwi_s1_chart_path, mndwi_s2_chart_path,
                        first_period=False, admin_info=None):
    """
    Append all content for one kelompok + one period to the story.

    Args:
        first_period : bool — if True, prepend the kelompok header and
                              administrative info (only needed on the first period)
        admin_info   : dict — {kabupaten, kecamatan, desa} or None
    """
    # ── Group header (only on the first period for this group) ──────────────
    if first_period:
        story.append(Paragraph(f"Kelompok: {name}", styles["section"]))
        story.append(HRFlowable(width="100%", thickness=2,
                                 color=colors.HexColor("#1a3c5e"), spaceAfter=6))

        # Administrative information table
        if admin_info:
            admin_val_style = ParagraphStyle(
                "CAdminVal", parent=styles["body"], fontSize=9, leading=13)
            admin_lbl_style = ParagraphStyle(
                "CAdminLbl", parent=styles["body"], fontSize=9, leading=13,
                fontName="Helvetica-Bold", textColor=colors.HexColor("#1a3c5e"))

            rows = []
            for label, val in [("Kabupaten", admin_info.get("kabupaten")),
                               ("Kecamatan", admin_info.get("kecamatan")),
                               ("Desa",      admin_info.get("desa"))]:
                if val is not None:
                    rows.append([
                        Paragraph(label, admin_lbl_style),
                        Paragraph(f": {val}", admin_val_style),
                    ])

            if rows:
                admin_tbl = Table(rows, colWidths=[3.0*cm, 13.5*cm])
                admin_tbl.setStyle(TableStyle([
                    ('VALIGN',        (0, 0), (-1, -1), 'TOP'),
                    ('TOPPADDING',    (0, 0), (-1, -1), 2),
                    ('BOTTOMPADDING', (0, 0), (-1, -1), 2),
                    ('LEFTPADDING',   (0, 0), (-1, -1), 0),
                    ('BACKGROUND',    (0, 0), (-1, -1), colors.HexColor("#f5f8fc")),
                    ('BOX',           (0, 0), (-1, -1), 0.4, colors.HexColor("#c8daea")),
                    ('LINEBELOW',     (0, 0), (-1, -2), 0.3, colors.HexColor("#dce9f5")),
                ]))
                story.append(admin_tbl)
                story.append(Spacer(1, 0.3*cm))

    # ── Period sub-header ───────────────────────────────────────────────────
    period_style = ParagraphStyle(
        "CPeriod", parent=styles["section"],
        fontSize=10, textColor=colors.HexColor("#3d6b8e"),
        spaceBefore=6, spaceAfter=4,
        backColor=colors.HexColor("#eef4fb"),
        borderPad=4,
    )
    story.append(Paragraph(f"&#128197;  Periode: {period_label}", period_style))
    story.append(Spacer(1, 0.2*cm))

    # ── Three index blocks ──────────────────────────────────────────────────
    _add_index_block(story, styles, 1,
                     "Normalized Difference Vegetation Index (NDVI)",
                     ndvi_map_path, ndvi_chart_path, ndvi_area)

    _add_index_block(story, styles, 2,
                     "MNDWI SWIR-1 (Modified Normalized Difference Water Index)",
                     mndwi_s1_map_path, mndwi_s1_chart_path, mndwi_s1_area)

    _add_index_block(story, styles, 3,
                     "MNDWI SWIR-2 (Modified Normalized Difference Water Index)",
                     mndwi_s2_map_path, mndwi_s2_chart_path, mndwi_s2_area)

    story.append(PageBreak())   # each period block ends on a new page

print("✓ Per-kelompok page builder defined")

---
## Step 8 — Main Pipeline

This is the orchestration function that ties everything together. It:

1. Initializes GEE and loads the asset
2. Loops over all groups (kelompok) and all time periods
3. For each combination, runs the full compute → download → render pipeline
4. Accumulates all content into the `story` list
5. Builds the final PDF

> **Performance note:** The main bottleneck is the `compute_class_areas()` calls, which each make a round-trip to GEE. With 74 groups × 2 periods × 3 indices = **444 GEE API calls**, expect ~20–40 minutes total runtime depending on your connection and GEE quota.

In [ ]:
def main():
    """
    Full pipeline: GEE → process → visualize → PDF.

    Temporary PNG files are written to a system temp directory and
    automatically deleted when the `with tempfile.TemporaryDirectory()`
    block exits — so the final PDF is all that remains.
    """
    # ── 1. Initialize GEE ───────────────────────────────────────────────────
    init_gee()

    # ── 2. Load the FeatureCollection asset ─────────────────────────────────
    print("Loading assets …")
    prm = ee.FeatureCollection(CONFIG["prm_asset"])

    # ── 3. Get the list of unique kelompok names ─────────────────────────────
    # aggregate_array retrieves all values of the field, distinct() deduplicates,
    # sort() orders alphabetically, and getInfo() fetches the result from GEE
    kelompok_field  = CONFIG["kelompok_field"]
    kelompok_values = (prm
                       .aggregate_array(kelompok_field)
                       .distinct()
                       .sort()
                       .getInfo())
    print(f"  Found {len(kelompok_values)} kelompok")

    # ── 4. Prepare PDF story and visualization parameters ───────────────────
    styles  = build_styles()
    story   = []
    build_cover_page(story, styles, kelompok_values)

    thumb_size  = CONFIG["thumb_width"]
    periods     = CONFIG["periods"]

    # GEE visualization dictionaries map class ID (1–4) to color palette
    ndvi_vis    = {"min": 1, "max": 4,
                   "palette": [c["hex"].replace("#","") for c in NDVI_LEGEND]}
    mndwi_vis   = {"min": 1, "max": 4,
                   "palette": [c["hex"].replace("#","") for c in MNDWI_LEGEND]}
    basemap_vis = {"bands": ["B4", "B3", "B2"],   # Red, Green, Blue
                   "min": 0, "max": 0.25, "gamma": 1.3}

    # ── 5. Process each kelompok ────────────────────────────────────────────
    # Use a temporary directory — all PNGs are auto-cleaned after PDF is built
    with tempfile.TemporaryDirectory() as tmpdir:

        for kidx, kname in enumerate(kelompok_values, start=1):
            print(f"\n[{kidx}/{len(kelompok_values)}] Kelompok: {kname}")

            # Filter the FeatureCollection to this group's features
            feat     = prm.filter(ee.Filter.eq(kelompok_field, kname))
            geometry = feat.geometry()   # union of all features for this group
            bounds   = get_bounds(geometry)
            safe     = str(kname).replace(" ", "_").replace("/", "-")  # safe filename

            # Extract administrative attributes from the first matching feature
            feat_props = feat.first().getInfo().get("properties", {})
            admin_info = {
                "kabupaten": feat_props.get(CONFIG.get("field_kabupaten", "kabupaten")),
                "kecamatan": feat_props.get(CONFIG.get("field_kecamatan", "kecamatan")),
                "desa":      feat_props.get(CONFIG.get("field_desa",      "desa")),
            }

            # ── Loop over each analysis period ─────────────────────────────
            for pidx, period in enumerate(periods):
                plabel = period["label"]
                pstart = period["start"]
                pend   = period["end"]
                p_safe = plabel.replace(" ", "_")
                print(f"  → {plabel} ({pstart} – {pend})")

                # Step A: Build the cloud-free Sentinel-2 composite
                s2 = build_sentinel2(geometry, pstart, pend)

                # Step B: Compute spectral indices
                ndvi, mndwi_s1, mndwi_s2 = compute_indices(s2)

                # Step C: Classify each index
                ndvi_cls     = classify_index(ndvi,     CONFIG["ndvi_thresholds"])
                mndwi_s1_cls = classify_index(mndwi_s1, CONFIG["mndwi_thresholds"])
                mndwi_s2_cls = classify_index(mndwi_s2, CONFIG["mndwi_thresholds"])

                # Step D: Compute area statistics (makes GEE API calls)
                print("    Computing areas …")
                ndvi_area     = compute_class_areas(ndvi_cls,     geometry, NDVI_LEGEND)
                mndwi_s1_area = compute_class_areas(mndwi_s1_cls, geometry, MNDWI_LEGEND)
                mndwi_s2_area = compute_class_areas(mndwi_s2_cls, geometry, MNDWI_LEGEND)

                # Step E: Download thumbnail images from GEE
                print("    Downloading thumbnails …")
                basemap_img      = download_image(get_thumb_url(s2,           geometry, basemap_vis, thumb_size))
                ndvi_cls_img     = download_image(get_thumb_url(ndvi_cls,     geometry, ndvi_vis,    thumb_size))
                mndwi_s1_cls_img = download_image(get_thumb_url(mndwi_s1_cls, geometry, mndwi_vis,   thumb_size))
                mndwi_s2_cls_img = download_image(get_thumb_url(mndwi_s2_cls, geometry, mndwi_vis,   thumb_size))

                # Step F: Render and save map PNGs to temp directory
                ndvi_map_path       = os.path.join(tmpdir, f"{safe}_{p_safe}_ndvi_map.png")
                mndwi_s1_map_path   = os.path.join(tmpdir, f"{safe}_{p_safe}_mndwi_s1_map.png")
                mndwi_s2_map_path   = os.path.join(tmpdir, f"{safe}_{p_safe}_mndwi_s2_map.png")
                ndvi_chart_path     = os.path.join(tmpdir, f"{safe}_{p_safe}_ndvi_chart.png")
                mndwi_s1_chart_path = os.path.join(tmpdir, f"{safe}_{p_safe}_mndwi_s1_chart.png")
                mndwi_s2_chart_path = os.path.join(tmpdir, f"{safe}_{p_safe}_mndwi_s2_chart.png")

                save_map_with_legend(basemap_img, ndvi_cls_img, NDVI_LEGEND,
                                     f"NDVI — {kname} ({plabel})", bounds, ndvi_map_path)
                save_map_with_legend(basemap_img, mndwi_s1_cls_img, MNDWI_LEGEND,
                                     f"MNDWI SWIR-1 — {kname} ({plabel})", bounds, mndwi_s1_map_path)
                save_map_with_legend(basemap_img, mndwi_s2_cls_img, MNDWI_LEGEND,
                                     f"MNDWI SWIR-2 — {kname} ({plabel})", bounds, mndwi_s2_map_path)

                save_area_barchart(ndvi_area,     f"NDVI — {plabel}",         ndvi_chart_path)
                save_area_barchart(mndwi_s1_area, f"MNDWI SWIR-1 — {plabel}", mndwi_s1_chart_path)
                save_area_barchart(mndwi_s2_area, f"MNDWI SWIR-2 — {plabel}", mndwi_s2_chart_path)

                # Step G: Add period block to the PDF story
                # first_period=True only on pidx==0 so the group header prints once
                build_kelompok_page(
                    story, styles, kname, plabel,
                    ndvi_area, mndwi_s1_area, mndwi_s2_area,
                    ndvi_map_path, mndwi_s1_map_path, mndwi_s2_map_path,
                    ndvi_chart_path, mndwi_s1_chart_path, mndwi_s2_chart_path,
                    first_period=(pidx == 0),
                    admin_info=admin_info,
                )
                print(f"    ✓ {plabel} added.")

        # ── 6. Build the PDF ─────────────────────────────────────────────────
        print(f"\nBuilding PDF → {CONFIG['output_pdf']} …")
        doc = SimpleDocTemplate(
            CONFIG["output_pdf"],
            pagesize=A4,
            leftMargin=MARGIN, rightMargin=MARGIN,
            topMargin=MARGIN,  bottomMargin=MARGIN,
        )
        doc.build(story)   # renders all flowables to PDF pages

    print(f"\n✅  Done!  Report saved to:  {CONFIG['output_pdf']}")


print("✓ main() defined — ready to run")

---
## Step 9 — Run the Pipeline

Everything is now defined. Run the cell below to execute the full pipeline.

**Before running**, double-check:
- [ ] `CONFIG["gee_project"]` is set to your GEE project ID
- [ ] `CONFIG["prm_asset"]` points to your FeatureCollection
- [ ] `CONFIG["kelompok_field"]` matches a real attribute field in your asset
- [ ] GEE authentication is set up (first run will open a browser)

> **Tip:** To test with a single group before running all 74, temporarily add a `.limit(1)` after loading the FeatureCollection in `main()`, or filter `kelompok_values = kelompok_values[:1]`.

In [ ]:
# Run the full pipeline
# Expected runtime: 20–40 minutes for 74 groups × 2 periods
if __name__ == "__main__":
    main()

---
## Summary & Next Steps

Congratulations! If the pipeline completed successfully, you now have a PDF report (`report_ndvi_mndwi.pdf`) containing:

- A cover page with metadata and legend definitions
- One section per community group (kelompok), showing:
  - Administrative info (kabupaten / kecamatan / desa)
  - For each analysis period: NDVI, MNDWI SWIR-1, and MNDWI SWIR-2 classified maps with overlaid basemap, bar charts, and area statistics tables

### Ways to extend this notebook

| Goal | How |
|---|---|
| **Add more periods** | Append more `{label, start, end}` dicts to `CONFIG["periods"]` |
| **Change classification thresholds** | Edit `CONFIG["ndvi_thresholds"]` and/or `CONFIG["mndwi_thresholds"]` and update the legend labels |
| **Use a different satellite** | Replace `'COPERNICUS/S2_SR_HARMONIZED'` with e.g. Landsat 8/9; adjust band names |
| **Export class maps to GEE Assets** | Use `ee.batch.Export.image.toAsset()` instead of thumbnail downloads |
| **Add change detection** | Compute the difference in class areas between periods and highlight trends |
| **Parallelize group processing** | Use Python `concurrent.futures.ThreadPoolExecutor` to process multiple groups simultaneously |
| **Download from Google Colab** | Run `from google.colab import files; files.download(CONFIG['output_pdf'])` after `main()` |

---

### Key concepts recap

```
NDVI  = (B8 − B4) / (B8 + B4)     →  vegetation density & health
MNDWI = (B3 − SWIR) / (B3 + SWIR) →  surface water extent

Cloud masking  → SCL band (Sentinel-2) removes clouds/shadows
Median reducer → robust aggregation across the time window
pixelArea()    → hectare calculation at 10 m native resolution
getThumbURL()  → GEE renders the classified map server-side → download PNG
ReportLab      → assembles PNGs + tables into a multi-page PDF
```